In [ ]:
# 1. Install necessary libraries
!pip install rasterio torch torchvision matplotlib tqdm

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Import standard libraries
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import rasterio
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

IMAGE_DIR = r"/content/drive/MyDrive/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/images"
LABEL_DIR = r"/content/drive/MyDrive/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/labels"

Mounted at /content/drive


In [ ]:
# --- CODE TO RUN AT THE START OF YOUR NOTEBOOK ---
!cp -r "/content/drive/MyDrive/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/images" /content/images_local
!cp -r "/content/drive/MyDrive/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/labels" /content/labels_local

# Update your config paths:
IMAGE_DIR = r"/content/images_local"
LABEL_DIR = r"/content/labels_local"

In [ ]:
from torch.utils.data import random_split

class SlumDataset(Dataset):
    def __init__(self, image_dir, label_dir, empty_ratio=0.2):
        self.image_dir = image_dir
        self.label_dir = label_dir

        # 1. Find all label files
        label_files = glob.glob(os.path.join(label_dir, "*.tif"))
        print(f"Checking {len(label_files)} candidate files...")

        self.valid_files = []
        self.empty_files = []

        # 2. Safety Check Loop
        for lbl_path in tqdm(label_files, desc="Verifying Pairs"):
            basename = os.path.basename(lbl_path)
            img_path = os.path.join(self.image_dir, basename)

            # CHECK: Does the matching image exist?
            if not os.path.exists(img_path):
                # If missing, skip silently
                continue

            # CHECK: Open label to see if it has data (slums) or is empty
            try:
                with rasterio.open(lbl_path) as src:
                    # Read a tiny preview (fast) to check for data
                    # We use a 1/10th scale overview for speed
                    data = src.read(1, out_shape=(1, int(src.height/10), int(src.width/10)))

                    if data.max() > 0:
                        self.valid_files.append(basename)
                    else:
                        self.empty_files.append(basename)
            except Exception as e:
                print(f"Error reading {basename}: {e}")

        # 3. Balancing (Mix some empty files in so the AI learns background too)
        num_empty = int(len(self.valid_files) * empty_ratio)
        selected_empty = random.sample(self.empty_files, min(num_empty, len(self.empty_files)))

        self.final_list = self.valid_files + selected_empty
        print(f"\nSUCCESS: Found {len(self.valid_files)} Slum images + {len(selected_empty)} Background images.")
        print(f"Total Dataset Size: {len(self.final_list)}")

    def __len__(self):
        return len(self.final_list)

    def __getitem__(self, idx):
        filename = self.final_list[idx]
        img_path = os.path.join(self.image_dir, filename)
        lbl_path = os.path.join(self.label_dir, filename)

        # Load Image (Input)
        with rasterio.open(img_path) as src:
            # Read and Normalize (0-255 -> 0.0-1.0)
            image = src.read().astype(np.float32) / 255.0

        # Load Label (Target)
        with rasterio.open(lbl_path) as src:
            label = src.read(1).astype(np.int64)

        return torch.from_numpy(image), torch.from_numpy(label)

# --- CREATE THE DATASET ---
dataset = SlumDataset(IMAGE_DIR, LABEL_DIR)

# --- SPLIT THE DATASET ---
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

train_subset, val_subset = random_split(dataset, [train_size, val_size])

print(f"Total Images: {total_size}")
print(f"Training on:  {train_size}")
print(f"Validating on: {val_size}")

NUM_WORKERS = 2 # Define NUM_WORKERS
train_loader = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_subset, batch_size=8, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

Checking 2361 candidate files...


Verifying Pairs:   0%|          | 0/2361 [00:00<?, ?it/s]


SUCCESS: Found 1497 Slum images + 299 Background images.
Total Dataset Size: 1796
Total Images: 1796
Training on:  1436
Validating on: 360


In [ ]:
!pip install segmentation-models-pytorch rasterio

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

class HybridLoss(nn.Module):
    def __init__(self, ce_weight=1.0, dice_weight=1.0):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss()
        self.dice_loss = smp.losses.DiceLoss(mode='multiclass')
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight

    def forward(self, y_pred, y_true):
        ce_l = self.ce_loss(y_pred, y_true)
        dice_l = self.dice_loss(y_pred, y_true)
        return (self.ce_weight * ce_l) + (self.dice_weight * dice_l)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import segmentation_models_pytorch as smp


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=2
).to(device)

# Use the new Hybrid Loss
criterion = HybridLoss(ce_weight=1.0, dice_weight=1.0)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Add a Learning Rate Scheduler:
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3 # Wait 3 epochs before reducing LR
)

def calculate_iou_for_slum_class(pr, gt, threshold=0.5):


    # Apply softmax to get probabilities
    probs = torch.softmax(pr, dim=1) # (B, C, H, W)


    preds = (probs[:, 1, :, :] > threshold).long() # (B, H, W), 0 or 1

    gt_slum = (gt == 1).long() # (B, H, W), 0 or 1

    # Calculate intersection
    intersection = (preds & gt_slum).sum().float()


    union = (preds | gt_slum).sum().float()


    iou = (intersection + 1e-7) / (union + 1e-7)
    return iou

metrics = {
    'iou_score': lambda pr, gt: \
        calculate_iou_for_slum_class(pr, gt, threshold=0.5)
}

In [ ]:
NUM_EPOCHS = 10
best_val_loss = float('inf')

train_losses = []
val_losses = []
val_ious = [] # New list for IoU metric


print(f"Starting Training on {device}...")

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)


    model.eval()
    val_loss = 0.0
    val_iou_score = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()

            # --- Calculate IoU/Jaccard Score (New) ---
            iou = metrics['iou_score'](outputs, masks)
            val_iou_score += iou.item()

    avg_val_loss = val_loss / len(val_loader)
    avg_val_iou = val_iou_score / len(val_loader)
    val_losses.append(avg_val_loss)
    val_ious.append(avg_val_iou)

    # --- Step 5: Update the Scheduler (New) ---
    scheduler.step(avg_val_loss)

    print(f"\n--> Epoch {epoch+1} Summary:")
    print(f"    Train Loss: {avg_train_loss:.4f}")
    # Report both Loss and the superior IoU metric
    print(f"    Valid Loss: {avg_val_loss:.4f} | Valid IoU (Slum): {avg_val_iou:.4f}")


    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print("        (Saved best model!)")

print("\nTraining Complete!")
print(f"Best Validation Loss: {best_val_loss:.4f}")

Starting Training on cuda...


Epoch 1/10 [Train]:   0%|          | 0/180 [00:00<?, ?it/s]


--> Epoch 1 Summary:
    Train Loss: 0.1429
    Valid Loss: 0.1851 | Valid IoU (Slum): 0.8376
        (Saved best model!)


Epoch 2/10 [Train]:   0%|          | 0/180 [00:00<?, ?it/s]


--> Epoch 2 Summary:
    Train Loss: 0.1438
    Valid Loss: 0.1815 | Valid IoU (Slum): 0.8406
        (Saved best model!)


...


--> Epoch 8 Summary:
    Train Loss: 0.1489
    Valid Loss: 0.1779 | Valid IoU (Slum): 0.8404


Epoch 9/10 [Train]:   0%|          | 0/180 [00:00<?, ?it/s]


--> Epoch 9 Summary:
    Train Loss: 0.1470
    Valid Loss: 0.1805 | Valid IoU (Slum): 0.8398


Epoch 10/10 [Train]:   0%|          | 0/180 [00:00<?, ?it/s]


--> Epoch 10 Summary:
    Train Loss: 0.1425
    Valid Loss: 0.1779 | Valid IoU (Slum): 0.8425

Training Complete!
Best Validation Loss: 0.1770


In [ ]:
pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.4 MB/s eta 0:00:00


In [ ]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import torch
import segmentation_models_pytorch as smp


TEST_IMAGE_PATH = "/tile_1.0.tif"
MODEL_WEIGHTS_PATH = "/content/best_model_optimized.pth(10)"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=2
).to(device)

try:
    model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
    model.eval() # Set the model to evaluation mode (turns off dropout/batchnorm)
    print("SUCCESS: Model weights loaded and model set to evaluation mode.")
except FileNotFoundError:
    print(f"ERROR: Model weights file not found at {MODEL_WEIGHTS_PATH}. Did you run the training block?")
    # Exit or raise error if the file is missing

SUCCESS: Model weights loaded and model set to evaluation mode.


In [ ]:
import torch
import segmentation_models_pytorch as smp
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=2
).to(device)

saved_model_path = "/content/best_model_optimized.pth(10)"

if os.path.exists(saved_model_path):
    model.load_state_dict(torch.load(saved_model_path, map_location=device))
    print(f"SUCCESS: Loaded weights from {saved_model_path}")
    print("Model is ready for testing!")
else:
    print(f"ERROR: Could not find {saved_model_path}. Please check the path.")


model.eval()

SUCCESS: Loaded weights from /content/best_model_optimized.pth(10)
Model is ready for testing!


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [ ]:
!pip install segmentation-models-pytorch

import torch
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
import glob # Import glob to find files
import segmentation_models_pytorch as smp # Import smp

# Define device to ensure it's available in this scope
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=2
).to(device)

# 2. Load your specific saved file
saved_model_path = "/content/best_model_optimized.pth(10)"

if os.path.exists(saved_model_path):
    model.load_state_dict(torch.load(saved_model_path, map_location=device))
    print(f"SUCCESS: Loaded weights from {saved_model_path}")
    print("Model is ready for testing!")
else:
    print(f"ERROR: Could not find {saved_model_path}. Please check the path.")

# 3. Switch to Evaluation Mode (Crucial!)
model.eval()

def test_single_image(image_path):
    # Check if file exists
    if not os.path.exists(image_path):
        print(f"File not found: {image_path}")
        return

    # 1. Load Image
    with rasterio.open(image_path) as src:
        # Read all bands
        full_img_data = src.read()

        # Ensure we only take the first 3 channels (RGB)
        if full_img_data.shape[0] >= 3:
            img_data = full_img_data[0:3, :, :]
        else:
            print(f"Warning: Image {os.path.basename(image_path)} has less than 3 channels ({full_img_data.shape[0]}). Skipping.")
            return
=
        img_data = img_data.astype(np.float32) / 255.0


    img_tensor = torch.from_numpy(img_data).unsqueeze(0).to(device)

    # 3. Predict
    with torch.no_grad():
        output = model(img_tensor)
        # Convert probabilities to Class 0 or 1
        prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()


    plt.figure(figsize=(12, 6))

    # Original Image
    plt.subplot(1, 2, 1)
    plt.imshow(img_data.transpose(1, 2, 0))
    plt.title(f"Satellite Input: {os.path.basename(image_path)}")
    plt.axis('off')


    plt.subplot(1, 2, 2)
    plt.imshow(prediction, cmap='gray')
    plt.title("AI Prediction (Loaded from Weights)")
    plt.axis('off')

    plt.show()

test_image_paths_list = [
   "/content/tile_1.0.tif",
  "/content/tile_1.10.tif",
   "/content/tile_1.16.tif",
   "/content/tile_1.88.tif",
   "/content/000002306.tif",
   "/content/tile_5.26.tif",
   "/content/tile_5.5.tif",
   "/content/tile_5.17.tif",
   "/content/tile_5.16.tif",
   "/content/tile_6.19.tif",
   "/content/satellite101.tif",
   "/content/tile_6.4.tif",
   "/content/tile_6.9.tif",
   "/content/tile_5.6.tif",
   "/content/satellite25.tif",
   "/content/satellite24.tif",
   "/content/satellite21.tif",
   "/content/000000026.tif",
   "/content/000000103.tif",
   "/content/000000033.tif",
   "/content/satellite22.tif",
   "/content/tile_5.14.tif",
   "/content/satellite11 (1).tif",
   "/content/satellite13 (1).tif",
   "/content/satellite14 .tif",
   "/content/satellite15.tif",
   "/content/satellite13.tif",
   "/content/satellite16.tif",
   "/content/satellite17.tif",
   "/content/satellite19.tif",
   "/content/satellite20.tif",
   "/content/000000452.tif",
   "/content/000002158.tif",
   "/content/000002285.tif",
   "/content/000002306.tif",
   "/content/000000943.tif"

]


for img_path in test_image_paths_list:
    test_single_image(img_path)

In [ ]:
import torch
import segmentation_models_pytorch as smp

# Ensure device is defined (from previous cells)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_eval = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=2
).to(device)


saved_model_path = '/content/best_model_optimized.pth(10)'
if os.path.exists(saved_model_path):
    model_eval.load_state_dict(torch.load(saved_model_path, map_location=device))
    print(f"Successfully loaded best model weights from {saved_model_path}")
else:
    print(f"ERROR: Best model weights not found at {saved_model_path}. Please ensure the training ran successfully and saved the model.")

model_eval.eval()



print("Evaluating model on the validation set...")
val_loss = 0.0
val_iou_score = 0.0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model_eval(images)
        loss = criterion(outputs, masks)
        val_loss += loss.item()

        iou = metrics['iou_score'](outputs, masks) # Use the defined IoU metric
        val_iou_score += iou.item()

avg_val_loss = val_loss / len(val_loader)
avg_val_iou = val_iou_score / len(val_loader)

print(f"\nEvaluation Results (from saved 'best_model.pth'):")
print(f"  Validation Loss: {avg_val_loss:.4f}")
print(f"  Validation IoU (Slum Class): {avg_val_iou:.4f}")


Successfully loaded best model weights from /content/best_model_optimized.pth(10)
Evaluating model on the validation set...

Evaluation Results (from saved 'best_model.pth'):
  Validation Loss: 0.2928
  Validation IoU (Slum Class): 0.7452


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def visualize_predictions(model, dataloader, device, num_samples=3):
    model.eval() # Set model to evaluation mode
    samples_shown = 0
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1) # Get probabilities
            predictions = torch.argmax(probabilities, dim=1) # Get class predictions

            for i in range(images.shape[0]):
                if samples_shown >= num_samples:
                    return

                fig, axes = plt.subplots(1, 3, figsize=(18, 6))


                axes[0].imshow(images[i].cpu().permute(1, 2, 0).numpy())
                axes[0].set_title('Original Image')
                axes[0].axis('off')

                # Ground Truth Mask
                axes[1].imshow(masks[i].cpu().numpy(), cmap='gray')
                axes[1].set_title('Ground Truth Mask')
                axes[1].axis('off')

                # Predicted Mask
                axes[2].imshow(predictions[i].cpu().numpy(), cmap='gray')
                axes[2].set_title('Predicted Mask')
                axes[2].axis('off')

                plt.tight_layout()
                plt.show()
                samples_shown += 1

# Visualize predictions for a few samples from the validation loader
print("Visualizing model predictions on validation samples:")
visualize_predictions(model_eval, val_loader, device, num_samples=100)
